In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from dotenv import load_dotenv
load_dotenv()
import os
import asyncio

api_key = os.getenv('OPENAI_API_KEY')
openai_model_client = OpenAIChatCompletionClient(
    model = "gpt-4o",
    api_key=api_key
)

In [2]:
dsa_solver = AssistantAgent(
    name = 'Complex_DSA_Solver',
    model_client=openai_model_client,
    description='A DSA solver',
    system_message="You give code in python to solve complex DSA problems. Give under 100 words"
)

code_reviewer = AssistantAgent(
    name = 'CODE_REVEIWER',
    model_client=openai_model_client,
    description='A Code Reviewer',
    system_message="You review the code given by the complex_dsa_solver and make sure it is optimized.Give under 10 words"
)

code_editor = AssistantAgent(
    name = 'CODE_EDITOR',
    model_client=openai_model_client,
    description='A Code editor',
    system_message="You make the code easy to understand and add comments wherever required.Give under 10 words"
)

In [7]:
from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer,code_editor],
    max_turns=3,

    )

async def test_team():
    task = TextMessage(content = "Write a code to sort array of 10 numbers",source='user')

    result = await team.run(task=task)
    i=1
    for each_agent_message in result.messages:
        print(i)
        print(f"{each_agent_message.source} : {each_agent_message.content}")
        print("\n\n")
        
        i+=1

await test_team()

1
user : Write a code to sort array of 10 numbers



2
Complex_DSA_Solver : Sure! Here's a simple Python code to sort an array of 10 numbers using the built-in `sorted()` function:

```python
numbers = [23, 1, 45, 34, 7, 2, 99, 55, 12, 37]
sorted_numbers = sorted(numbers)
print("Sorted array:", sorted_numbers)
```

This code sorts the list `numbers` in ascending order and prints the sorted array.



3
CODE_REVEIWER : Efficient code using Python's built-in sorting function.



4
CODE_EDITOR : Utilizes efficient, built-in Python sort function effectively.





In [4]:
await team.reset()  # Reset the team for a new task.
from autogen_agentchat.base import TaskResult

async for message in team.run_stream(task="Write a simple Hello world code ?"):  # type: ignore
    if isinstance(message, TaskResult):
        print("Stop Reason:", message.stop_reason)
    else:
        print(message.source,message)

user source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 6, 17, 19, 30, 47, 707655, tzinfo=datetime.timezone.utc) content='Write a simple Hello world code ?' type='TextMessage'
Complex_DSA_Solver source='Complex_DSA_Solver' models_usage=RequestUsage(prompt_tokens=36, completion_tokens=47) metadata={} created_at=datetime.datetime(2025, 6, 17, 19, 30, 49, 472174, tzinfo=datetime.timezone.utc) content='Here\'s a simple "Hello, World!" program in Python:\n\n```python\nprint("Hello, World!")\n```\n\nThis single line of code uses the `print()` function to output the text "Hello, World!" to the console.' type='TextMessage'
CODE_REVEIWER source='CODE_REVEIWER' models_usage=RequestUsage(prompt_tokens=99, completion_tokens=9) metadata={} created_at=datetime.datetime(2025, 6, 17, 19, 30, 50, 75645, tzinfo=datetime.timezone.utc) content='Code is optimal and concise for task given.' type='TextMessage'
CODE_EDITOR source='CODE_EDITOR' models_usage=RequestUsage(prompt_token

In [8]:
from autogen_agentchat.ui import Console
await team.reset()  # Reset the team for a new task.
await Console(team.run_stream(task="Write a simple Hello world code."))  # Stream the messages to the console.


---------- TextMessage (user) ----------
Write a simple Hello world code.
---------- TextMessage (Complex_DSA_Solver) ----------
Here is a simple "Hello, World!" program in Python:

```python
print("Hello, World!")
```

This code uses the `print` function to display the string "Hello, World!" to the console.
---------- TextMessage (CODE_REVEIWER) ----------
Code is already optimized and simple.
---------- TextMessage (CODE_EDITOR) ----------
Great! The "Hello, World!" code is clear and concise.


TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 35, 55, 17579, tzinfo=datetime.timezone.utc), content='Write a simple Hello world code.', type='TextMessage'), TextMessage(source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=36, completion_tokens=45), metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 35, 55, 882374, tzinfo=datetime.timezone.utc), content='Here is a simple "Hello, World!" program in Python:\n\n```python\nprint("Hello, World!")\n```\n\nThis code uses the `print` function to display the string "Hello, World!" to the console.', type='TextMessage'), TextMessage(source='CODE_REVEIWER', models_usage=RequestUsage(prompt_tokens=97, completion_tokens=7), metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 35, 56, 754662, tzinfo=datetime.timezone.utc), content='Code is already optimized and simple.', type='TextMessage'), TextMessage(source='CODE_EDITOR', models_usage=Reque

In [6]:
await Console(team.run_stream(task="conitnue."))

---------- TextMessage (user) ----------
conitnue.
---------- TextMessage (CODE_REVEIWER) ----------
Ready to assist whenever you need!
---------- TextMessage (CODE_EDITOR) ----------
Absolutely! Here whenever you need assistance.
---------- TextMessage (Complex_DSA_Solver) ----------
Thank you! I'm here to help whenever you need further assistance.
---------- TextMessage (CODE_REVEIWER) ----------
Happy to help whenever you need it!
---------- TextMessage (CODE_EDITOR) ----------
Ready to assist whenever you need help!
---------- TextMessage (Complex_DSA_Solver) ----------
Thank you! I'm here whenever you need assistance.
---------- TextMessage (CODE_REVEIWER) ----------
Let me know how else I can help!


TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 31, 0, 507035, tzinfo=datetime.timezone.utc), content='conitnue.', type='TextMessage'), TextMessage(source='CODE_REVEIWER', models_usage=RequestUsage(prompt_tokens=234, completion_tokens=7), metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 31, 1, 343310, tzinfo=datetime.timezone.utc), content='Ready to assist whenever you need!', type='TextMessage'), TextMessage(source='CODE_EDITOR', models_usage=RequestUsage(prompt_tokens=252, completion_tokens=8), metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 31, 1, 866811, tzinfo=datetime.timezone.utc), content='Absolutely! Here whenever you need assistance.', type='TextMessage'), TextMessage(source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=254, completion_tokens=13), metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 31, 2, 573979, tzinfo=datetime.timezone.utc), content="Tha

In [11]:
from autogen_agentchat.agents import AssistantAgent

assistant1 = AssistantAgent(
    name = "assistant_1",
    model_client=openai_model_client,
    system_message="Add 1 to the number, first number is 0. Give the result as output."
)

assistant2 = AssistantAgent(
    name = "assistant_2",
    model_client=openai_model_client,
    system_message="Add 1 to the number you got from previous run. Give the result as output."
)
assistant3 = AssistantAgent(
    name = "assistant_3",
    model_client=openai_model_client,
    system_message="Add 1 to the number you got from previous run. Give the result as output."
)

In [12]:
my_increment_team = RoundRobinGroupChat(participants=[assistant1,assistant2,assistant3],max_turns=3)
await Console(my_increment_team.run_stream())

---------- TextMessage (assistant_1) ----------
The result of adding 1 to the number 0 is 1.
---------- TextMessage (assistant_2) ----------
The result of adding 1 to the number 1 is 2.
---------- TextMessage (assistant_3) ----------
The result of adding 1 to the number 2 is 3.


TaskResult(messages=[TextMessage(source='assistant_1', models_usage=RequestUsage(prompt_tokens=26, completion_tokens=15), metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 44, 32, 957789, tzinfo=datetime.timezone.utc), content='The result of adding 1 to the number 0 is 1.', type='TextMessage'), TextMessage(source='assistant_2', models_usage=RequestUsage(prompt_tokens=48, completion_tokens=15), metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 44, 34, 572225, tzinfo=datetime.timezone.utc), content='The result of adding 1 to the number 1 is 2.', type='TextMessage'), TextMessage(source='assistant_3', models_usage=RequestUsage(prompt_tokens=71, completion_tokens=15), metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 44, 35, 125513, tzinfo=datetime.timezone.utc), content='The result of adding 1 to the number 2 is 3.', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

In [13]:
await Console(my_increment_team.run_stream())

---------- TextMessage (assistant_1) ----------
The result of adding 1 to the number 3 is 4.
---------- TextMessage (assistant_2) ----------
The result of adding 1 to the number 4 is 5.
---------- TextMessage (assistant_3) ----------
The result of adding 1 to the number 5 is 6.


TaskResult(messages=[TextMessage(source='assistant_1', models_usage=RequestUsage(prompt_tokens=91, completion_tokens=15), metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 45, 34, 609833, tzinfo=datetime.timezone.utc), content='The result of adding 1 to the number 3 is 4.', type='TextMessage'), TextMessage(source='assistant_2', models_usage=RequestUsage(prompt_tokens=113, completion_tokens=15), metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 45, 35, 301344, tzinfo=datetime.timezone.utc), content='The result of adding 1 to the number 4 is 5.', type='TextMessage'), TextMessage(source='assistant_3', models_usage=RequestUsage(prompt_tokens=136, completion_tokens=15), metadata={}, created_at=datetime.datetime(2025, 6, 17, 19, 45, 36, 877899, tzinfo=datetime.timezone.utc), content='The result of adding 1 to the number 5 is 6.', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')